<a href="https://colab.research.google.com/github/dannroldan/Procesos-estocasticos/blob/main/ACT5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Ejercicio 3: Aproximación con $M$ fijo

El teorema principal de uniformización nos dice que la matriz de transición a tiempo continuo $P(t)$ se puede calcular mediante una serie infinita que pondera las potencias de la matriz discreta $\hat{P}$ usando probabilidades de Poisson.

Para implementarlo en la compu, no podemos sumar hasta el infinito. Este ejercicio propone truncar la serie en un número de términos $M$ fijo, el cual se estima para capturar la mayor parte de la masa de probabilidad de Poisson:
$$M \approx \max\{rt + 5\sqrt{rt}, 20\}$$

Además, verificamos una de las propiedades de las CMTC: la **Ecuación de Chapman-Kolmogorov**. Esta ecuación dice que calcular la transición para un tiempo $t=1$ debe ser exactamente equivalente a hacer la transición a la mitad del tiempo ($t=0.5$) y luego multiplicarla por otra transición de $t=0.5$.

Es decir:
$$P(1) = P(0.5)P(0.5)$$

In [12]:
import numpy as np
import math

#definir la matriz de tasas
R = np.array([
    [0, 2, 3, 0],
    [4, 0, 2, 0],
    [0, 2, 0, 2],
    [1, 0, 3, 0]
], dtype=float)

r = 6.0
N = R.shape[0]

#matriz estocástica P gorrito
P_hat = np.copy(R) / r
for i in range(N):
    r_i = np.sum(R[i])
    P_hat[i, i] = 1.0 - (r_i / r)

#función para aproximar P(t) con M fijo
def calc_P_fixed_M(t, r, P_hat):
    rt = r * t
    #cálculo de M
    M = max(int(rt + 5 * math.sqrt(rt)), 20)

    P_t = np.zeros((N, N))
    P_k = np.eye(N)

    for k in range(M + 1):
        poisson_prob = math.exp(-rt) * (rt**k) / math.factorial(k)
        P_t += poisson_prob * P_k
        P_k = np.dot(P_k, P_hat)

    return P_t, M

#cálculos para t = 0.5, 1 y 5
P_05_fixed, M_05 = calc_P_fixed_M(0.5, r, P_hat)
P_1_fixed, M_1 = calc_P_fixed_M(1.0, r, P_hat)
P_5_fixed, M_5 = calc_P_fixed_M(5.0, r, P_hat)

print("Ejercicio 3")
print(f"M utilizado para t=0.5: {M_05}")
print(f"M utilizado para t=1.0: {M_1}")
print(f"M utilizado para t=5.0: {M_5}")

print("\nMatriz P(0.5):")
print(np.round(P_05_fixed, 5))

#Chapman-Kolmogorov
print("\nChapman-Kolmogorov")
P_05_por_P_05 = np.dot(P_05_fixed, P_05_fixed)

print("P(1) calculada normal:")
print(np.round(P_1_fixed, 5))

print("\nP(0.5) * P(0.5):")
print(np.round(P_05_por_P_05, 5))

son_iguales = np.allclose(P_1_fixed, P_05_por_P_05, atol=1e-5)
print(f"\nSe cumple la igualdad P(1) = P(0.5)P(0.5)? {son_iguales}")

Ejercicio 3
M utilizado para t=0.5: 20
M utilizado para t=1.0: 20
M utilizado para t=5.0: 57

Matriz P(0.5):
[[0.25061 0.21696 0.38666 0.14577]
 [0.25313 0.23836 0.37441 0.13409]
 [0.16912 0.19361 0.4203  0.21696]
 [0.15802 0.15744 0.39833 0.28621]]

Chapman-Kolmogorov
P(1) calculada normal:
[[0.20615 0.2039  0.39871 0.19124]
 [0.20828 0.20534 0.3979  0.18847]
 [0.19676 0.19838 0.40096 0.2039 ]
 [0.19205 0.194   0.40147 0.21248]]

P(0.5) * P(0.5):
[[0.20615 0.2039  0.39871 0.19124]
 [0.20828 0.20534 0.3979  0.18847]
 [0.19676 0.19838 0.40096 0.2039 ]
 [0.19205 0.194   0.40147 0.21248]]

Se cumple la igualdad P(1) = P(0.5)P(0.5)? True




### Ejercicio 4.2: Algoritmo de Uniformización con Tolerancia $\epsilon$

El algoritmo optimiza el cálculo determinando los términos de la distribución de Poisson de manera recursiva e iterativa usando la siguiente propiedad matemática:
$$c_{nuevo} = c_{viejo} \cdot \frac{rt}{k}$$

En esta formulación, la variable $c$ representa el cálculo iterativo del término $e^{-rt} \frac{(rt)^k}{k!}$.Al mismo tiempo, se implementa una variable acumuladora (`sum`) que registra la masa de probabilidad procesada hasta la iteración actual

En lugar de definir un número de iteraciones $M$ de manera arbitraria, el ciclo se detiene dinámicamente cuando la suma de las probabilidades procesadas alcanza el valor de $1 - \epsilon$.
Por el teorema de cotas de error por truncamiento, esta condición de paro garantiza que el error residual de la matriz aproximada es estrictamente menor a la tolerancia $\epsilon$ especificada.

In [13]:
#función usando el algoritmo de uniformización con tolerancia epsilon
def calc_P_uniformization(t, r, P_hat, epsilon=0.00001):

    rt = r * t

    A = np.copy(P_hat)
    B = math.exp(-rt) * np.eye(N)
    c = math.exp(-rt)
    suma = c
    k = 1

    #iteración mientras el error sea mayor a epsilon
    while suma < (1.0 - epsilon):
        c = c * (rt) / k
        B = B + c * A
        A = np.dot(A, P_hat)
        suma = suma + c
        k = k + 1

    return B, k - 1

print("Ejercicio 4.2 (Tolerancia epsilon = 0.00001)")

tiempos = [0.5, 1.0, 5.0]

for t in tiempos:
    P_eps, M_eps = calc_P_uniformization(t, r, P_hat, epsilon=0.00001)

    _, M_fixed = calc_P_fixed_M(t, r, P_hat)

    print(f"\nPara t = {t}:")
    print(f" M usado por la fórmula (Ej 3): {M_fixed}")
    print(f" M usado por el algoritmo epsilon (Ej 4.2): {M_eps}")
    print("  Matriz resultante P(t):")
    print(np.round(P_eps, 5))

Ejercicio 4.2 (Tolerancia epsilon = 0.00001)

Para t = 0.5:
 M usado por la fórmula (Ej 3): 20
 M usado por el algoritmo epsilon (Ej 4.2): 13
  Matriz resultante P(t):
[[0.25061 0.21696 0.38666 0.14577]
 [0.25313 0.23836 0.37441 0.13409]
 [0.16912 0.19361 0.4203  0.21696]
 [0.15802 0.15744 0.39833 0.28621]]

Para t = 1.0:
 M usado por la fórmula (Ej 3): 20
 M usado por el algoritmo epsilon (Ej 4.2): 19
  Matriz resultante P(t):
[[0.20615 0.2039  0.39871 0.19124]
 [0.20828 0.20534 0.3979  0.18847]
 [0.19676 0.19838 0.40096 0.2039 ]
 [0.19205 0.194   0.40147 0.21248]]

Para t = 5.0:
 M usado por la fórmula (Ej 3): 57
 M usado por el algoritmo epsilon (Ej 4.2): 56
  Matriz resultante P(t):
[[0.2 0.2 0.4 0.2]
 [0.2 0.2 0.4 0.2]
 [0.2 0.2 0.4 0.2]
 [0.2 0.2 0.4 0.2]]


##Comparación
Al observar los resultados que nos arrojó el código para los diferentes tiempos ($t = 0.5, 1.0, 5.0$), podemos notar:
1. El algoritmo de epsilon es mucho más eficiente para tiempos cortos. Por ejemplo, al calcular $t = 0.5$, la fórmula del ej. 3 nos obligaba a hacer 20 pasos. En cambio, el algoritmo nuevo se dio cuenta de que ya había alcanzado la precisión necesaria y se detuvo en 13 pasos. Para tiempos largos (como $t = 5.0$), ambos métodos hacen casi el mismo esfuerzo (57 vs 56). Básicamente el algoritmo del Ejercicio 4.2 es mejor porque trabaja solo lo estrictamente necesario.
2. Si vemos los números de las matrices resultantes, podemos ver:
A corto plazo ($t = 0.5$ y $t = 1.0$): Los números de las probabilidades cambian según el renglón que se vea. Esto significa que el estado en el que iniciaste todavía importa. El sistema aún "recuerda" su punto de inicio.
A largo plazo ($t = 5.0$):Todos los renglones de la matriz quedaron iguales: [0.2, 0.2, 0.4, 0.2]. Esto nos demuestra que el sistema ya se estabilizó y alcanzó su distribución límite.
Básicamente después de un buen rato (cuando llegamos a $t=5.0$), al sistema ya se le "olvidó" por completo en q estado comenzó. Sin importar su punto de partida, sabemos que el sistema pasará el 20% del tiempo en el estado 1, 20% en el estado 2, 40% en el estado 3 y 20% en el estado 4.
Ese es su equilibrio final.